# 🎯 NSE Sector-Wise Stock Data Fetcher

This notebook lets you select specific sectors and fetch their stock data to Google Sheets.

## Features
- ✅ Interactive dropdown to select any sector
- ✅ Multi-select to fetch multiple sectors at once
- ✅ Auto-upload to Google Sheets with formatting
- ✅ Preview data before uploading

## Available Sectors
- NIFTY AUTO, NIFTY BANK, NIFTY IT, NIFTY PHARMA
- NIFTY FMCG, NIFTY METAL, NIFTY REALTY, NIFTY ENERGY
- NIFTY FINANCIAL SERVICES, NIFTY MEDIA, NIFTY PSU BANK
- NIFTY PRIVATE BANK, NIFTY HEALTHCARE INDEX
- NIFTY CONSUMER DURABLES, NIFTY OIL & GAS

---

## Step 1: Install Libraries

In [ ]:
# Install required libraries
!pip install -q nsepythonserver gspread google-auth google-auth-oauthlib google-auth-httplib2 pandas tqdm ipywidgets

print("✅ All libraries installed!")

## Step 2: Import Libraries

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm
import ipywidgets as widgets
from IPython.display import display, clear_output

# Import nsepythonserver
from nsepythonserver import *

# Import Google Sheets libraries
from google.colab import auth
import gspread
from google.auth import default

print("✅ Libraries imported!")

## Step 3: Configuration

In [ ]:
# Configuration
SHEET_NAME = "NSE Sector Stocks"  # 👈 Change this to your desired sheet name

# All available sectors
ALL_SECTORS = [
    'NIFTY AUTO',
    'NIFTY BANK',
    'NIFTY IT',
    'NIFTY PHARMA',
    'NIFTY FMCG',
    'NIFTY METAL',
    'NIFTY REALTY',
    'NIFTY ENERGY',
    'NIFTY FINANCIAL SERVICES',
    'NIFTY MEDIA',
    'NIFTY PSU BANK',
    'NIFTY PRIVATE BANK',
    'NIFTY HEALTHCARE INDEX',
    'NIFTY CONSUMER DURABLES',
    'NIFTY OIL & GAS'
]

print(f"📋 Configuration complete!")
print(f"📊 Sheet Name: {SHEET_NAME}")
print(f"🏭 Available sectors: {len(ALL_SECTORS)}")

## Step 4: Helper Functions

In [ ]:
def fetch_sector_stocks(sector_name):
    """Fetch stocks for a specific sector"""
    try:
        quote = nse_get_index_quote(sector_name.lower())
        if quote and 'data' in quote:
            stocks = quote['data']
            df = pd.DataFrame(stocks)
            if not df.empty:
                df['Sector'] = sector_name
                return df
        return pd.DataFrame()
    except Exception as e:
        print(f"⚠️ Error fetching {sector_name}: {e}")
        return pd.DataFrame()

def upload_to_sheets(df, worksheet_name):
    """Upload DataFrame to Google Sheets"""
    if df.empty:
        print("⚠️ No data to upload")
        return None
    
    try:
        # Authenticate
        print("🔐 Authenticating...")
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)
        
        # Get or create spreadsheet
        try:
            spreadsheet = gc.open(SHEET_NAME)
        except gspread.SpreadsheetNotFound:
            spreadsheet = gc.create(SHEET_NAME)
        
        # Get or create worksheet
        try:
            worksheet = spreadsheet.worksheet(worksheet_name)
            worksheet.clear()
        except gspread.WorksheetNotFound:
            worksheet = spreadsheet.add_worksheet(
                title=worksheet_name,
                rows=len(df) + 100,
                cols=len(df.columns) + 5
            )
        
        # Upload data
        data = [df.columns.tolist()] + df.values.tolist()
        worksheet.update('A1', data)
        
        # Format header
        worksheet.format('A1:Z1', {
            'backgroundColor': {'red': 0.26, 'green': 0.52, 'blue': 0.96},
            'textFormat': {'bold': True, 'foregroundColor': {'red': 1, 'green': 1, 'blue': 1}},
            'horizontalAlignment': 'CENTER'
        })
        worksheet.freeze(rows=1)
        
        return spreadsheet.url
    except Exception as e:
        print(f"❌ Error uploading: {e}")
        return None

print("✅ Helper functions defined!")

---
# 🎯 OPTION 1: Select Single Sector

Choose one sector from the dropdown and fetch its stocks.

---

In [ ]:
# Create dropdown
sector_dropdown = widgets.Dropdown(
    options=ALL_SECTORS,
    value='NIFTY AUTO',
    description='Select Sector:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

# Create button
fetch_button = widgets.Button(
    description='📊 Fetch Stocks',
    button_style='success',
    layout=widgets.Layout(width='200px')
)

# Output area
output = widgets.Output()

def on_fetch_click(b):
    with output:
        clear_output()
        sector = sector_dropdown.value
        
        print(f"\n{'='*70}")
        print(f"📊 Fetching stocks for: {sector}")
        print(f"{'='*70}\n")
        
        # Fetch data
        df = fetch_sector_stocks(sector)
        
        if not df.empty:
            print(f"✅ Fetched {len(df)} stocks\n")
            print("📋 Preview (first 10 stocks):")
            print(df.head(10).to_string())
            
            # Upload to sheets
            print(f"\n📤 Uploading to Google Sheets...")
            sheet_name = sector.replace(' ', '_')
            url = upload_to_sheets(df, sheet_name)
            
            if url:
                print(f"\n{'='*70}")
                print(f"✅ SUCCESS!")
                print(f"{'='*70}")
                print(f"📊 Uploaded {len(df)} stocks to '{sheet_name}'")
                print(f"🔗 Open: {url}")
        else:
            print(f"⚠️ No stocks found for {sector}")

fetch_button.on_click(on_fetch_click)

# Display UI
print("🎯 SELECT A SECTOR:")
display(widgets.VBox([
    sector_dropdown,
    fetch_button,
    output
], layout=widgets.Layout(padding='10px')))

---
# 🎨 OPTION 2: Select Multiple Sectors

Choose multiple sectors and fetch all their stocks in one go!

**Tip:** Hold `Ctrl` (Windows) or `Cmd` (Mac) to select multiple sectors.

---

In [ ]:
# Create multi-select
sector_multiselect = widgets.SelectMultiple(
    options=ALL_SECTORS,
    value=['NIFTY AUTO', 'NIFTY IT'],
    description='Select Sectors:',
    style={'description_width': '120px'},
    rows=10,
    layout=widgets.Layout(width='400px')
)

# Create button
fetch_multi_button = widgets.Button(
    description='📊 Fetch Selected Sectors',
    button_style='info',
    layout=widgets.Layout(width='250px')
)

# Output area
output_multi = widgets.Output()

def on_fetch_multi_click(b):
    with output_multi:
        clear_output()
        sectors = list(sector_multiselect.value)
        
        if not sectors:
            print("⚠️ Please select at least one sector!")
            return
        
        print(f"\n{'='*70}")
        print(f"📊 Fetching {len(sectors)} sectors")
        print(f"{'='*70}\n")
        
        all_stocks = []
        
        for sector in tqdm(sectors, desc="Fetching"):
            df = fetch_sector_stocks(sector)
            if not df.empty:
                # Remove Index column if exists
                if 'Index' in df.columns:
                    df = df.drop('Index', axis=1)
                all_stocks.append(df)
                print(f"✅ {sector}: {len(df)} stocks")
            else:
                print(f"⚠️ {sector}: No data")
        
        if all_stocks:
            combined_df = pd.concat(all_stocks, ignore_index=True)
            
            print(f"\n📋 Total: {len(combined_df)} stocks")
            print(f"\n📊 Breakdown by sector:")
            print(combined_df['Sector'].value_counts())
            
            # Upload to sheets
            print(f"\n📤 Uploading to Google Sheets...")
            url = upload_to_sheets(combined_df, "Selected_Sectors")
            
            if url:
                print(f"\n{'='*70}")
                print(f"✅ SUCCESS!")
                print(f"{'='*70}")
                print(f"📊 Uploaded {len(combined_df)} stocks from {len(sectors)} sectors")
                print(f"📄 Sheet Name: Selected_Sectors")
                print(f"🔗 Open: {url}")
        else:
            print(f"\n⚠️ No stock data collected")

fetch_multi_button.on_click(on_fetch_multi_click)

# Display UI
print("🎨 SELECT MULTIPLE SECTORS (Hold Ctrl/Cmd):")
display(widgets.VBox([
    sector_multiselect,
    fetch_multi_button,
    output_multi
], layout=widgets.Layout(padding='10px')))

---

## 📝 Notes

- **Authentication**: You'll be prompted to authenticate with Google the first time
- **Sheet Location**: Your Google Sheet will appear in Google Drive
- **Formatting**: Headers are automatically formatted with blue background
- **Updates**: Run the cells again to refresh data

## 💡 Tips

- **NIFTY AUTO**: Get all automotive sector stocks
- **NIFTY IT**: Get all IT sector stocks
- **NIFTY BANK**: Get all banking sector stocks
- **Multiple sectors**: Great for comparing stocks across sectors

---

**Disclaimer:** This tool is for informational purposes only. Always verify data from official sources before making investment decisions.
